First we sanity check the corpus...
Let's look at the stories generated and visually ensure they make sense.

In [1]:
import random
import textwrap
import sys

from pathlib import Path

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

CORPUS_DIR = PROJECT_ROOT / "datasets" / "qwen-emotion-stories" / "corpus"

from core.shards import read_shards
from core.utils import emotion_words_named

/Users/folusoogunlana/code/oss/emotion-concepts/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [29]:
from collections import Counter

rows = read_shards(CORPUS_DIR, sample=False)
print(f"{len(rows)} stories")
print(Counter(r["emotion"] for r in rows))

807 stories
Counter({'neutral': 98, 'surprised': 91, 'ashamed': 68, 'disgusted': 65, 'desperate': 64, 'afraid': 59, 'sad': 59, 'calm': 58, 'angry': 54, 'proud': 52, 'excited': 50, 'joyful': 48, 'lonely': 41})


In [3]:
def show_samples(n: int = 2, seed: int | None = None, emotion: str | None = None, sample: bool = True, prompts: bool = True) -> None:
    rows = read_shards(CORPUS_DIR, sample=sample)
    if emotion:
        rows = [r for r in rows if r["emotion"] == emotion]

    for row in random.Random(seed).sample(rows, min(n, len(rows))):
        named = emotion_words_named(row["text"])
        print("=" * 100)
        print(f"{row['emotion']} | {row['topic']} | #{row['index']} | {len(row['text'].split())} words | ends with: {row['text'].rstrip()[-1]!r} | names: {named or 'none'}")
        if prompts:
            print("-" * 100)
            print(textwrap.indent(row["prompt"], "  "))
        print("-" * 100)
        print(textwrap.fill(row["text"], width=88))
        print()

show_samples(2, sample=False)

afraid | Someone's childhood imaginary friend appears in their niece's drawings | #1 | 71 words | ends with: '.' | names: none
----------------------------------------------------------------------------------------------------
  Write a short story (roughly one paragraph) based on the following premise.

  Topic: Someone's childhood imaginary friend appears in their niece's drawings

  The story should follow a character who is feeling afraid.

  Write the story in English. Use either third-person or first-person narration.

  Write between 90 and 130 words. Finish the final sentence. Do not write a title.

  The character is ALREADY feeling afraid in the very first sentence. Open inside the scene, at the moment the feeling is strongest. Do not begin with backstory, scene-setting, or a build-up towards the feeling, and do not begin with "Once upon a time".

  ONE STATE ONLY: the character feels afraid and nothing else, from the first word to the last. Nothing in the story relieves, re

Next up we need to extract activations from all layers to do our difference of means probe

In [4]:
## get activations

from core.models import Model

m = Model()
m.load_weights()

Loading weights: 100%|██████████| 290/290 [00:02<00:00, 114.47it/s]


In [5]:
BATCH_SIZE = 64
DEVICE = m.device

In [6]:
import torch

from collections import defaultdict

# Extract activations
# Create a function for extracting activations
# The function should take in a model and a list of texts, and it should return a list of tensors of activations
# For each prompt, there should be an activation with the shape of the residual stream
# Split the texts into train and test before passing into the function - only pass train
# TODO: skip until 20 or 18
def extract_activations(texts: list[str], batch_size: int = BATCH_SIZE, layers: list[int] = None):
    # iterate over texts (they will be specific to an emotion)
    # for each text, teacher force into model and get hidden states
    # get hidden states in each layer and mean-pool them
    # return activations
    m.tok.padding_side = "right"
    final_states = defaultdict(list)

    if layers is None:
        layers = list(range(m.model.config.num_hidden_layers + 1))

    for b in range(0, len(texts), batch_size):
        inputs = m.tok(texts[b : b + batch_size], padding=True, return_tensors='pt', truncation=False).to(DEVICE)

        with torch.no_grad():
            outputs = m.model.model(**inputs, output_hidden_states=True)
        
        mask = inputs.attention_mask.unsqueeze(-1)
        # shape: b, seq, 1

        for l in layers:
            hidden_states = (outputs.hidden_states[l] * mask).sum(1) / mask.sum(1)
            # shape: b, d_model
            
            final_states[l].append(hidden_states)

    return {l: torch.cat(v, dim=0) for l, v in final_states.items()}

activations = extract_activations([r["text"] for r in rows[:BATCH_SIZE]])
activations[0].shape


torch.Size([64, 896])

In [30]:
# Emotion vectors
# Create a function for creating emotion vectors
# It should take an emotion and all the texts, and it should spit out the emotion vector
# It should take the mean of all the train activations for each emotion - e_mean
# It should take the mean of all activations completely - mean
# It should subtract e_mean - mean to find the emotion vector

def emotion_vectors(texts: list[str], batch_size: int = BATCH_SIZE):
    # for each emotion (apart from neutral)
        # get all the train texts for the specific emotion
        # get activations for those texts
        # store mean activations for emotion
    # subtract out mean of all emotions
    # return list of emotion vectors
    counts = Counter(r["emotion"] for r in rows)
    emotions = sorted(e for e in counts if e != "neutral")
    vectors = {}

    train = [t for t in texts if t["split"] == "train"]
    acts = extract_activations([t["text"] for t in train], batch_size)
    global_mean = {}

    for l in range(len(acts)):
        global_mean[l] = acts[l].mean(0)

    for e in emotions:
        train_texts = [t for t in train if t["emotion"] == e]
        train_acts = extract_activations([t["text"] for t in train_texts])
        for l in range(len(train_acts)):
            pooled = train_acts[l].mean(0)
            # shape: d_model
            vectors[(e, l)] = pooled - global_mean[l]

    return vectors        

emovecs = emotion_vectors(rows)
emovecs[('afraid', 0)].shape

torch.Size([896])

In [31]:
# Denoising
# Create a function to denoise the emotion vectors in line with the paper
# To denoise do PCA on the set of the neutral activations
# Then after that, take the top components explaining 50% of variance and project them out from the emotion vectors
# Fit with PCA once per layer
# How do I confirm they have been denoised?? - 
    # Test with pairwise cosine similarity before and after. Also test with nearest centroid accuracy on held-out topics before vs after

def denoise(vecs: torch.Tensor, neutrals: list[str], batch_size: int = BATCH_SIZE, frac = 0.5, layers: list[int] = None):
    # teacher force the neutrals to get their activations
    # do PCA on the neutral activations 
    # measure variance using eigenvalues as weights
    # take the first n vectors until 0.5 of eigenvalue density
    # project the neutral activations onto those n vectors (to get the right magnitude of noise)
    # do the above with a matrix multiplication
    # subtract out the projected neutral activations from the emotion vectors
    # return the emotion vectors
    emotions = sorted(list(set([e for (e, l) in vecs.keys()])))
    acts = extract_activations(neutrals, batch_size)
    denoised = {}

    if layers is None:
        layers = range(len(acts))

    for l in layers:
        X = acts[l].cpu().float()
        # shape: n, d_model

        Xc = X - X.mean(0)
        _, S, Vh = torch.linalg.svd(Xc, full_matrices=False)
        ratio = (S ** 2) / (S ** 2).sum()
        # shape: n

        k = int((ratio.cumsum(dim=0) < frac).sum().item()) + 1
        pcs = Vh[:k].to(DEVICE)

        for e in emotions:
            v = vecs[(e, l)]
            denoised[(e, l)] = v - ((v @ pcs.T) @ pcs)
        
    return denoised

denoised_emovecs = denoise(emovecs, [r["text"] for r in rows if r["emotion"] == "neutral"])

In [32]:
emotions = sorted(list(set([e for (e, l) in denoised_emovecs.keys()])))
emotions

['afraid',
 'angry',
 'ashamed',
 'calm',
 'desperate',
 'disgusted',
 'excited',
 'joyful',
 'lonely',
 'proud',
 'sad',
 'surprised']

In [33]:
# sanity check denoising
# check that cosine similarity of noisy and noiseless vectors are not close to 0 or 1.0
# if cosine similarity is close to 1.0, then the denoising did little
# if cosine similarity is close to 0.0, then there's a problem with the denoising, or the original vectors were completely noise

cosines = []
for key in denoised_emovecs.keys():
    cosines.append(torch.nn.functional.cosine_similarity(emovecs[key], denoised_emovecs[key], dim=-1))

sum(cosines) / len(cosines)

tensor(0.6387, device='mps:0')

In [34]:
# logit lens
# the basic idea of the logit lens is to apply an unembedding matrix to the output
# the unembedding matrix should output high probabilities on tokens that match the emotion concept
# this is a crude verification technique but should work
import polars as pl

START_LAYER = 10

def logit_lens(vecs: torch.Tensor, k: int = 10) -> torch.Tensor:
    # first thing is to get the unembedding matrix... how? - can use the models one but double-check that's correct
    # HOLD ON - this should take the vectors, not the activations!!!
    # activations: n, seq, d_model, vecs: d_model
    # apply unembedding matrix to each vector for each emotion, layer combination
    # interesting... how to figure out which layer is the right layer??? - analysis to be done here
    # right now we just un-embed all layers after middle
    # decode tokens from logits{}
    # print out top_n most probable tokens per layer

    W_U = m.model.get_output_embeddings().weight
    # shape: vocab, d_model
    # norm = m.model.model.norm

    top_k = defaultdict(list)
    for (e, l), v in vecs.items():
        if l < START_LAYER: # start from the mid-layers
            continue

        h = v.to(W_U.device, W_U.dtype)
        logits = h @ W_U.T
        # shape: vocab
        top_k_ids = logits.topk(k).indices
        top_k_tokens = [m.tok.decode(i) for i in top_k_ids]
        top_k[e].insert(0, ",".join(top_k_tokens))
    
    return top_k
        
lens = logit_lens(denoised_emovecs)

In [37]:
pp_lens = {}
for e, tokens in lens.items():
    pp_lens[e] = tokens

df = pl.DataFrame(pp_lens)
df.head()

afraid,angry,ashamed,calm,desperate,disgusted,excited,joyful,lonely,proud,sad,surprised
str,str,str,str,str,str,str,str,str,str,str,str
"""拼命,用力,痉, panicked,紧张,紧紧,紧,剧烈,尖…",""" slammed, slamming, fury,爆炸,拳头…","""汗水, fatigue,无力, unable,缩, inad…",""" tranqu, gentle, relaxed,宁静, s…","""绝望,拳头,无力,狠,用力,硬,拼命,溃, agony, f…","""溃, nause, vom,恶心, gag,废气, spit…","""兴奋,激动,澎湃, excitement, thrilled…",""" sunshine, delight, delighted,…",""" silence, empt,寂寞,寂静,沉默, melan…",""" proudly, proud, victorious, r…",""" sorrow, melanch,侵蚀,溃, despair…","""…”,.” ,,”, alarmed, panicked, …"
"""eña, fists,抓紧,攥,趑, thro,обыти,…","""硬,rage,愤,狠, slammed,拳头, fury,狰…","""]-$,eña,✕,regor,UsageId,趑,øre,…",""" tranqu,恬,宁静,oothing,ipa,流水, r…","""硬, fists,أوض,攥, syll, desperat…",""" gag,硬,唾,秽,厉,臭,肛,狰, bile, bitt…","""兴奋,激动,澎湃,怦, excitement,跃, exci…",""" dew, delight,跳跃, sunshine,洒,I…","""寂寞, silence,穿透,沉默, empt, ?>', …",""" trium,自豪, proud,illum,胜利,自信, …","""侵蚀,]-$,ickness,穿透,Echo, melanc…","""…”,…” , trembling,,__,hazi,;!…"
"""рев, thro,甲状,أوض,₫, ><?, tremb…","""愤,硬,狠,拳头,rage, fists,撕, slamme…","""]-$,无力,ласт, grips,缩,.She,不住,e…","""柔和,宁静, tranqu,ipa, relaxed, br…","""硬,邶,甲状,딪, thro,أوض,ไ,�单,胛,喀""","""唾,硬,耻, gag,腥,秽,厉, bile,撕,邶""","""兴奋,激动,怦,澎湃,��이, excitement,跃, …",""" sunshine, delight,阳光, joy, ha…",""" silence,寂寞,unset, shadows, em…","""胜利,自信,flush, proud, trium,自豪, …","""侵蚀,]-$,ickness,涩,穿透,滴滴,-cloud,…",""" trembling, trem,curities,警惕,i…"
"""أوض,обыти, trem,紧,рев,خش, nerv…","""狠,狰,硬,愤, slammed, violently, F…","""但她,.She,ласт,UsageId, herself,…","""宁静,柔和, relaxing,悠闲, tranqu,惬意,…","""硬,狰,[SerializeField,邶,巯,螵,أوض,…","""溃,硬,耻,黏, gag, muc, epith,聱,肛, …","""激动,兴奋,心跳, excitement, adrenali…",""" delight,享受, joy, sunshine,快乐,…",""" empt,寂寞,.She,)(__,mute, shado…","""自信, smile,胜利,自豪, positivity, t…","""ickness,℅,侵蚀,.mybatis, unread,…",""" trembling,curities, unread, t…"
"""خش,紧,.Suppress,笾,أوض,ael,обыти…","""硬,狠,愤,垃,追い,狰, PROFITS,脓,噻,Form…","""ласт,却不,ahrungen,不肯,UsageId,无力…","""宁静,柔和, breeze, gently, relaxin…",""" thói,[SerializeField,噻,邶,ἴ,ὑ,…",""" thói,ッ,硬,噻,聱,肛,เย,脓,溃,ὑ""","""兴奋,��이,跃,激动,addOn,每一个,霎,刹那,躍,澎…",""" delight,/stretch,享受,,""); ,跃,;…","""]-$, empt,mute,寂寞,流逝, untouche…","""自豪, satisfied,胜利,风采, triumph,绽…","""]-$,ickness,.mybatis,[Serializ…",""" suddenly,无知,iac, mActivity,惑,…"


In [ ]:
# six-way accuracy